# NLP Assignment 3: Transformers + RAG
**Student ID**: i23XXXX  
**Course**: CS-4063 Natural Language Processing  
**Due Date**: 29-04-26

## Instructions
1. Ensure the dataset files (`Software_5.json.gz`, etc.) are in the root directory.
2. This notebook implements an Encoder-only Transformer for multi-task sentiment and category classification, and a Decoder-only Transformer for RAG-enhanced explanation generation.
3. All Transformer components are implemented from scratch using basic PyTorch layers.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import gzip
import json
import random
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, Dataset
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Transformer Implementation (Scratch)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model, self.num_heads, self.d_k = d_model, num_heads, d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        q = self.W_q(q).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(k).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(v).view(bs, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return self.W_o(torch.matmul(attn, v).transpose(1, 2).contiguous().view(bs, -1, self.d_model))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.attn(x, x, x, mask)))
        return self.norm2(x + self.dropout(self.ff(x)))

class MultiTaskEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, num_sentiment=3, num_category=3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.s_head = nn.Linear(d_model, num_sentiment)
        self.c_head = nn.Linear(d_model, num_category)
    def forward(self, x, mask=None):
        x = self.pos(self.emb(x))
        for layer in self.layers: x = layer(x, mask)
        pooled = torch.mean(x, dim=1)
        return self.s_head(pooled), self.c_head(pooled), pooled

class CausalTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)
    def forward(self, x, mask=None):
        x = self.pos(self.emb(x))
        for layer in self.layers: x = layer(x, mask)
        return self.fc_out(x)

## 2. Load Preprocessed Data

In [ ]:
data = torch.load('data/processed_data.pt')
vocab = data['vocab']
train_X, train_Ys, train_Yc, train_texts, train_summs = data['train']
test_X, test_Ys, test_Yc, test_texts, test_summs = data['test']
print(f"Vocab Size: {len(vocab)}")

## 3. Retrieval Module

In [ ]:
class RetrievalModule:
    def __init__(self, train_embeddings, train_texts):
        self.train_embeddings = train_embeddings
        self.train_texts = train_texts
    def retrieve(self, query_emb, k=1):
        if query_emb.dim() == 1: query_emb = query_emb.unsqueeze(0)
        sims = F.cosine_similarity(query_emb, self.train_embeddings)
        scores, idxs = torch.topk(sims, k)
        return [{'text': self.train_texts[i.item()], 'score': s.item()} for s, i in zip(scores, idxs)]

## 4. Evaluation & Results

In [ ]:
def generate(model, prompt, max_len=40, min_len=10, temperature=0.7, top_k=50):
    model.eval()
    tokens = [vocab.get(t, vocab['<unk>']) for t in prompt.split()]
    input_ids = torch.tensor([vocab['<sos>']] + tokens).unsqueeze(0).to(device)
    generated = []
    for step in range(max_len):
        mask = torch.tril(torch.ones(input_ids.size(1), input_ids.size(1))).to(device)
        with torch.no_grad():
            logits = model(input_ids, mask=mask)[:, -1, :]
            for token in set(generated[-5:]): logits[0, token] /= 1.5 # Repetition penalty
            logits = logits / temperature
            if step < min_len: logits[0, vocab['<eos>']] = float('-inf')
            v, i = torch.topk(logits, top_k)
            next_token = i[0, torch.multinomial(torch.softmax(v, dim=-1), 1)].item()
        if next_token == vocab['<eos>']: break
        generated.append(next_token)
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]]).to(device)], dim=1)
    return " ".join([{v:k for k,v in vocab.items()}[idx] for idx in generated])
